# Spell Feature Engineering

This notebook extracts features from D&D 5e spell data.

**Input:** `data/spells.csv`
**Output:** Updated `data/spells.csv` with additional feature columns

## Imports and Setup

In [56]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

# Detect execution context
cwd = Path.cwd()
if cwd.name == 'notebooks':
    DATA_DIR = '../data'
    sys.path.insert(0, '.')
    from helper_files import parse_spell_targets, calculate_average_damage
else:
    DATA_DIR = './data'
    sys.path.insert(0, '.')
    from notebooks.helper_files import parse_spell_targets, calculate_average_damage

print(f"Data directory: {DATA_DIR}")
print("Imports successful")

Data directory: ./data
Imports successful


## Configuration

Tunable parameters for DPR calculations. Adjust these to observe different model behaviors.

In [57]:
# =============================================================================
# DPR CALCULATION PARAMETERS
# =============================================================================
# Adjust these values to tune how spell DPR is calculated

# AOE_PASSTHROUGH_MULTIPLIER: Scales the damage for AoE spells
# - 1.0 = full damage passes through (default, no modification)
# - 0.5 = AoE damage counts as 50% effective (e.g., accounting for saves)
# - Use this to model the "effective" damage of AoE vs single-target
AOE_PASSTHROUGH_MULTIPLIER = 1.0

# TARGET_CAP: Maximum number of targets used in modified_dpr calculation
# - None = no cap, use full estimated_targets (default)
# - 4 = cap at 4 targets (common design assumption for balanced encounters)
# - 2 = very conservative cap
# Example: Fireball estimates 7.5 targets, but with TARGET_CAP=4, uses 4
TARGET_CAP = 4

print(f"DPR Configuration:")
print(f"  AOE_PASSTHROUGH_MULTIPLIER = {AOE_PASSTHROUGH_MULTIPLIER}")
print(f"  TARGET_CAP = {TARGET_CAP if TARGET_CAP else 'None (no cap)'}")

DPR Configuration:
  AOE_PASSTHROUGH_MULTIPLIER = 1.0
  TARGET_CAP = 4


In [58]:
def calculate_modified_dpr(row, aoe_multiplier=AOE_PASSTHROUGH_MULTIPLIER, target_cap=TARGET_CAP):
    """
    Calculate modified DPR for a spell based on base damage and targets.
    
    Parameters:
    -----------
    row : pd.Series
        Row from spell dataframe with 'avg_damage', 'is_aoe', 'estimated_targets', 'target_count'
    aoe_multiplier : float
        Multiplier applied to AoE spell damage (default from config)
    target_cap : int or None
        Maximum number of targets to count (default from config)
    
    Returns:
    --------
    float : modified_dpr value
    
    Formula:
    --------
    For AoE spells:
        effective_targets = min(estimated_targets, target_cap) if target_cap else estimated_targets
        modified_dpr = base_dpr × effective_targets × aoe_multiplier
    
    For non-AoE spells:
        modified_dpr = base_dpr × target_count (no multiplier applied)
    """
    base_dpr = row['avg_damage'] if pd.notna(row['avg_damage']) else 0
    
    if base_dpr == 0:
        return 0.0
    
    if row['is_aoe']:
        # AoE spell: use estimated_targets with cap and multiplier
        targets = row['estimated_targets'] if pd.notna(row['estimated_targets']) else 1
        if target_cap is not None:
            targets = min(targets, target_cap)
        return base_dpr * targets * aoe_multiplier
    else:
        # Non-AoE spell: use target_count directly (no multiplier)
        targets = row['target_count'] if pd.notna(row['target_count']) else 1
        if target_cap is not None:
            targets = min(targets, target_cap)
        return base_dpr * targets

print("DPR calculation function defined")

DPR calculation function defined


## Load Spell Data

In [59]:
load_path = DATA_DIR + '/spells.csv'
df = pd.read_csv(load_path)

In [60]:
# Parse target information for each spell
target_data = df['description'].apply(parse_spell_targets)

# Extract individual columns from the dict results
df['target_count'] = target_data.apply(lambda x: x['target_count'])
df['is_aoe'] = target_data.apply(lambda x: x['is_aoe'])
df['aoe_type'] = target_data.apply(lambda x: x['aoe_type'])
df['aoe_size'] = target_data.apply(lambda x: x['aoe_size'])
df['estimated_targets'] = target_data.apply(lambda x: x['estimated_targets'])

# For non-AoE spells with target_count, use that as estimated_targets
df.loc[df['target_count'].notna() & df['estimated_targets'].isna(), 'estimated_targets'] = df['target_count']

print("Target features extracted")

Target features extracted


## Calculate DPR Columns

Using the configurable parameters to calculate:
- `base_dpr`: Raw average damage (same as avg_damage)
- `effective_targets`: Targets used in calculation (respects TARGET_CAP)
- `modified_dpr`: base_dpr × effective_targets × AOE_PASSTHROUGH_MULTIPLIER (for AoE)

In [61]:
# Calculate DPR columns
df['base_dpr'] = df['avg_damage'].fillna(0)

# Calculate effective targets (respects TARGET_CAP)
def get_effective_targets(row):
    if row['is_aoe']:
        targets = row['estimated_targets'] if pd.notna(row['estimated_targets']) else 1
    else:
        targets = row['target_count'] if pd.notna(row['target_count']) else 1
    
    if TARGET_CAP is not None:
        targets = min(targets, TARGET_CAP)
    return targets

df['effective_targets'] = df.apply(get_effective_targets, axis=1)

# Calculate modified DPR
df['modified_dpr'] = df.apply(calculate_modified_dpr, axis=1)

# Show example calculations
print("=== DPR Calculation Examples ===")
print(f"Config: AOE_PASSTHROUGH_MULTIPLIER={AOE_PASSTHROUGH_MULTIPLIER}, TARGET_CAP={TARGET_CAP}")
print()

examples = ['Fireball', 'Magic Missile', 'Cone of Cold', 'Scorching Ray', 'Eldritch Blast']
for spell_name in examples:
    spell = df[df['spell_name'] == spell_name]
    if len(spell) > 0:
        row = spell.iloc[0]
        aoe_marker = "(AoE)" if row['is_aoe'] else ""
        print(f"{spell_name} {aoe_marker}:")
        print(f"  base_dpr={row['base_dpr']:.1f}, effective_targets={row['effective_targets']:.1f}, modified_dpr={row['modified_dpr']:.1f}")

=== DPR Calculation Examples ===
Config: AOE_PASSTHROUGH_MULTIPLIER=1.0, TARGET_CAP=4

Fireball (AoE):
  base_dpr=28.0, effective_targets=4.0, modified_dpr=112.0
Magic Missile :
  base_dpr=3.5, effective_targets=3.0, modified_dpr=10.5
Cone of Cold (AoE):
  base_dpr=36.0, effective_targets=4.0, modified_dpr=144.0
Scorching Ray :
  base_dpr=7.0, effective_targets=3.0, modified_dpr=21.0
Eldritch Blast :
  base_dpr=5.5, effective_targets=1.0, modified_dpr=5.5


## Extract Condition Features

Parse spell descriptions to identify which conditions they can inflict.
Mirrors the monster features: `inflicts_blinded`, `inflicts_charmed`, etc.

In [62]:
import re

# =============================================================================
# CONDITION FEATURES (matches monster feature naming)
# =============================================================================
# List of conditions to detect (same as 1_feature_engineering.ipynb)
CONDITIONS = [
    'blinded', 'charmed', 'deafened', 'frightened', 'incapacitated',
    'paralyzed', 'petrified', 'poisoned', 'prone', 'restrained', 'stunned'
]

def extract_condition_features(description):
    """
    Extract condition infliction features from spell description.
    Returns dict with inflicts_{condition} boolean flags.
    """
    if pd.isna(description):
        return {f'inflicts_{c}': False for c in CONDITIONS}
    
    desc_lower = description.lower()
    result = {}
    
    for condition in CONDITIONS:
        # Patterns that indicate the spell inflicts this condition:
        # - "target is [condition]"
        # - "becomes [condition]"
        # - "be [condition]"
        # - "creature is [condition]"
        # - "knocked prone" / "falls prone"
        # - "[condition] for the duration"
        
        patterns = [
            rf'\b{condition}\b',  # Basic presence of the condition word
        ]
        
        # Check if any pattern matches
        found = any(re.search(p, desc_lower) for p in patterns)
        
        # Exclude false positives: spells that REMOVE or PREVENT conditions
        # e.g., "can't be charmed", "immunity to being frightened"
        if found:
            exclusion_patterns = [
                rf"can't be {condition}",
                rf"cannot be {condition}",
                rf"immune to .+{condition}",
                rf"immunity to .+{condition}",
                rf"ends? .+{condition}",
                rf"cured? of .+{condition}",
                rf"no longer {condition}",
            ]
            if any(re.search(p, desc_lower) for p in exclusion_patterns):
                found = False
        
        result[f'inflicts_{condition}'] = found
    
    return result

# Apply to all spells
condition_features = df['description'].apply(extract_condition_features).apply(pd.Series)
already_joined = False
for col in condition_features.columns:
    if col in df.columns:
        already_joined = True
if already_joined == False:
    df = pd.concat([df, condition_features], axis=1)

# Summary
print("=== Condition Features ===")
for condition in CONDITIONS:
    col = f'inflicts_{condition}'
    count = df[col].sum()
    if count > 0:
        print(f"  {col}: {count} spells")

# Show examples
print("\n=== Example Spells with Conditions ===")
condition_examples = [
    ('Hold Person', 'paralyzed'),
    ('Fear', 'frightened'),
    ('Blindness/Deafness', 'blinded'),
    ('Tasha\'s Hideous Laughter', 'prone'),
    ('Command', 'prone'),
]
for spell_name, expected_condition in condition_examples:
    spell = df[df['spell_name'] == spell_name]
    if len(spell) > 0:
        row = spell.iloc[0]
        col = f'inflicts_{expected_condition}'
        print(f"  {spell_name}: {col}={row[col]}")

=== Condition Features ===
  inflicts_blinded: 17 spells
  inflicts_charmed: 22 spells
  inflicts_deafened: 6 spells
  inflicts_frightened: 11 spells
  inflicts_incapacitated: 12 spells
  inflicts_paralyzed: 5 spells
  inflicts_poisoned: 4 spells
  inflicts_prone: 12 spells
  inflicts_restrained: 9 spells
  inflicts_stunned: 6 spells

=== Example Spells with Conditions ===
  Hold Person: inflicts_paralyzed=True
  Fear: inflicts_frightened=True
  Blindness/Deafness: inflicts_blinded=True
  Tasha's Hideous Laughter: inflicts_prone=True
  Command: inflicts_prone=False


## Extract Buff/Debuff Features

Parse spell descriptions to identify:
- `grants_flying`: Spells that grant flying speed (Fly, Levitate, etc.)
- `ac_bonus`: AC bonus granted (Shield = +5, Shield of Faith = +2)
- `attack_bonus`: Attack roll bonus (Bless = avg +2.5)
- `save_bonus`: Saving throw bonus
- `grants_advantage`: Grants advantage on attacks/saves
- `inflicts_disadvantage`: Inflicts disadvantage on target

In [63]:
# =============================================================================
# BUFF/DEBUFF FEATURES
# =============================================================================

def extract_buff_debuff_features(description):
    """
    Extract buff/debuff features from spell description.
    Returns dict with flying, AC bonus, attack bonus, advantage/disadvantage.
    """
    if pd.isna(description):
        return {
            'grants_flying': False,
            'ac_bonus': 0,
            'attack_bonus': 0.0,
            'save_bonus': 0.0,
            'grants_advantage': False,
            'inflicts_disadvantage': False
        }
    
    desc_lower = description.lower()
    result = {}
    
    # =========================================================================
    # GRANTS FLYING
    # =========================================================================
    flying_patterns = [
        r'flying speed',
        r'gains? a flying',
        r'target can fly',
        r'creature can fly',
        r'you can fly',
        r'ability to fly',
    ]
    result['grants_flying'] = any(re.search(p, desc_lower) for p in flying_patterns)
    
    # =========================================================================
    # AC BONUS
    # =========================================================================
    ac_bonus = 0
    
    # Direct bonus: "+5 bonus to AC"
    ac_match = re.search(r'\+(\d+)\s+bonus to ac', desc_lower)
    if ac_match:
        ac_bonus = int(ac_match.group(1))
    
    # Mage Armor style: "AC becomes 13 + Dex modifier"
    mage_armor_match = re.search(r'ac becomes (\d+)\s*\+', desc_lower)
    if mage_armor_match and ac_bonus == 0:
        base_ac = int(mage_armor_match.group(1))
        ac_bonus = max(0, base_ac - 10)
    
    result['ac_bonus'] = ac_bonus
    
    # =========================================================================
    # ATTACK BONUS
    # =========================================================================
    attack_bonus = 0.0
    
    # Bless style: "roll a d4 and add the number rolled to the attack roll"
    if re.search(r'd4.+add.+(?:to the )?attack roll', desc_lower):
        attack_bonus = 2.5
    
    # Direct bonus: "+X to attack rolls"
    attack_match = re.search(r'\+(\d+)\s+(?:bonus\s+)?to\s+attack\s+rolls?', desc_lower)
    if attack_match:
        attack_bonus = float(attack_match.group(1))
    
    result['attack_bonus'] = attack_bonus
    
    # =========================================================================
    # SAVE BONUS
    # =========================================================================
    save_bonus = 0.0
    
    # Bless style: "roll a d4 and add... to... saving throw"
    if re.search(r'd4.+add.+saving throw', desc_lower):
        save_bonus = 2.5
    
    # Direct bonus
    save_match = re.search(r'\+(\d+)\s+(?:bonus\s+)?to\s+saving\s+throws?', desc_lower)
    if save_match:
        save_bonus = float(save_match.group(1))
    
    result['save_bonus'] = save_bonus
    
    # =========================================================================
    # GRANTS ADVANTAGE
    # =========================================================================
    advantage_patterns = [
        r'(?:has|have|gains?)\s+advantage\s+on\s+(?:attack|weapon)',
        r'attack rolls?.+have advantage',
        r'advantage on.+attack rolls?',
        r'attack roll.+has advantage',  # Faerie Fire style
    ]
    result['grants_advantage'] = any(re.search(p, desc_lower) for p in advantage_patterns)
    
    # =========================================================================
    # INFLICTS DISADVANTAGE
    # =========================================================================
    disadvantage_patterns = [
        r'(?:has|have)\s+disadvantage\s+on\s+(?:attack|weapon)',
        r'attack rolls?.+have disadvantage',
        r'disadvantage on.+attack rolls?',
        r'(?:has|have)\s+disadvantage\s+on\s+ability\s+checks',
        r'(?:has|have)\s+disadvantage\s+on\s+(?:strength|dexterity|constitution|intelligence|wisdom|charisma)',
    ]
    result['inflicts_disadvantage'] = any(re.search(p, desc_lower) for p in disadvantage_patterns)
    
    return result

# Apply to all spells
buff_features = df['description'].apply(extract_buff_debuff_features).apply(pd.Series)
df = pd.concat([df, buff_features], axis=1)

# Summary
print("=== Buff/Debuff Features ===")
print(f"  grants_flying: {df['grants_flying'].sum()} spells")
print(f"  ac_bonus > 0: {(df['ac_bonus'] > 0).sum()} spells")
print(f"  attack_bonus > 0: {(df['attack_bonus'] > 0).sum()} spells")
print(f"  save_bonus > 0: {(df['save_bonus'] > 0).sum()} spells")
print(f"  grants_advantage: {df['grants_advantage'].sum()} spells")
print(f"  inflicts_disadvantage: {df['inflicts_disadvantage'].sum()} spells")

# Show examples
print("\n=== Example Buff/Debuff Spells ===")
buff_examples = [
    ('Fly', 'grants_flying'),
    ('Shield', 'ac_bonus'),
    ('Shield of Faith', 'ac_bonus'),
    ('Bless', 'attack_bonus'),
    ('Faerie Fire', 'grants_advantage'),
    ('Bane', 'inflicts_disadvantage'),
]
for spell_name, feature in buff_examples:
    spell = df[df['spell_name'] == spell_name]
    if len(spell) > 0:
        row = spell.iloc[0]
        print(f"  {spell_name}: {feature}={row[feature]}")

=== Buff/Debuff Features ===
  grants_flying: grants_flying    10
grants_flying    10
dtype: int64 spells
  ac_bonus > 0: ac_bonus    8
ac_bonus    8
dtype: int64 spells
  attack_bonus > 0: attack_bonus    3
attack_bonus    3
dtype: int64 spells
  save_bonus > 0: save_bonus    3
save_bonus    3
dtype: int64 spells
  grants_advantage: grants_advantage    37
grants_advantage    37
dtype: int64 spells
  inflicts_disadvantage: inflicts_disadvantage    32
inflicts_disadvantage    32
dtype: int64 spells

=== Example Buff/Debuff Spells ===
  Fly: grants_flying=grants_flying    True
grants_flying    True
Name: 266, dtype: object
  Shield: ac_bonus=ac_bonus    5
ac_bonus    5
Name: 121, dtype: object
  Shield of Faith: ac_bonus=ac_bonus    2
ac_bonus    2
Name: 122, dtype: object
  Bless: attack_bonus=attack_bonus    2.5
attack_bonus    2.5
Name: 58, dtype: object
  Faerie Fire: grants_advantage=grants_advantage    True
grants_advantage    True
Name: 83, dtype: object
  Bane: inflicts_disadvant

## Target Analysis

In [64]:
# Check specific spells to verify parsing
examples = [
    'Magic Missile',      # Should be 3 targets (darts)
    'Fireball',           # Should be AoE radius ~7.5 targets
    'Cure Wounds',        # Should be 1 target
    'Scorching Ray',      # Should be 3 targets (rays)
    'Lightning Bolt',     # Should be AoE line ~3 targets
    'Cone of Cold',       # Should be AoE cone ~10.8 targets
    'Hold Person',        # Should be 1 target
    'Chain Lightning',    # Should be multi-target or AoE
    'Eldritch Blast',     # Should be 1 target (at base level)
    'Burning Hands',      # Should be AoE cone (small)
]

print("=== Example Spells ===")
for spell_name in examples:
    spell = df[df['spell_name'] == spell_name]
    if len(spell) > 0:
        row = spell.iloc[0]
        if row['is_aoe']:
            print(f"{spell_name}: AoE {row['aoe_type']} ({row['aoe_size']}) → ~{row['estimated_targets']} targets")
        elif pd.notna(row['target_count']):
            print(f"{spell_name}: {int(row['target_count'])} target(s)")
        else:
            print(f"{spell_name}: unknown targeting")

=== Example Spells ===
Magic Missile: 3 target(s)
Fireball: AoE radius (20-foot) → ~7.5 targets
Cure Wounds: 1 target(s)
Scorching Ray: 3 target(s)
Lightning Bolt: AoE line (100-foot) → ~3.0 targets
Cone of Cold: AoE cone (60-foot) → ~10.8 targets
Hold Person: 1 target(s)
Chain Lightning: 3 target(s)
Eldritch Blast: 1 target(s)
Burning Hands: AoE cone (15-foot) → ~1.0 targets


In [65]:
# AoE type breakdown
print("\n=== AoE Types ===")
aoe_spells = df[df['is_aoe']]
print(aoe_spells['aoe_type'].value_counts())


=== AoE Types ===
aoe_type
radius    80
cube      35
cone      17
area      12
line       9
square     8
Name: count, dtype: int64


In [66]:
# Target count distribution (non-AoE)
print("\n=== Target Count Distribution (non-AoE) ===")
non_aoe = df[~df['is_aoe'] & df['target_count'].notna()]
print(non_aoe['target_count'].value_counts().sort_index())


=== Target Count Distribution (non-AoE) ===
target_count
1.0     316
2.0       1
3.0       8
4.0       2
5.0       4
6.0       3
8.0       4
10.0      6
Name: count, dtype: int64


## Verify Examples

In [67]:
# Check specific spells to verify parsing
examples = [
    'Magic Missile',      # Should be 3 targets (darts)
    'Fireball',           # Should be AoE radius
    'Cure Wounds',        # Should be 1 target
    'Scorching Ray',      # Should be 3 targets (rays)
    'Lightning Bolt',     # Should be AoE line
    'Cone of Cold',       # Should be AoE cone
    'Hold Person',        # Should be 1 target
    'Chain Lightning',    # Should be multi-target or AoE
    'Eldritch Blast',     # Should be 1 target (at base level)
]

print("=== Example Spells ===")
for spell_name in examples:
    spell = df[df['spell_name'] == spell_name]
    if len(spell) > 0:
        row = spell.iloc[0]
        if row['is_aoe']:
            print(f"{spell_name}: AoE ({row['aoe_type']}, {row['aoe_size']})")
        elif pd.notna(row['target_count']):
            print(f"{spell_name}: {int(row['target_count'])} target(s)")
        else:
            print(f"{spell_name}: unknown targeting")

=== Example Spells ===
Magic Missile: 3 target(s)
Fireball: AoE (radius, 20-foot)
Cure Wounds: 1 target(s)
Scorching Ray: 3 target(s)
Lightning Bolt: AoE (line, 100-foot)
Cone of Cold: AoE (cone, 60-foot)
Hold Person: 1 target(s)
Chain Lightning: 3 target(s)
Eldritch Blast: 1 target(s)


# Top single-target damage spells


In [68]:
damage_spells = df[df['avg_damage']>0].reset_index(drop=True)

In [69]:
print("\\n=== Top 10 Single-Target Damage Spells ===")
top_single = damage_spells[damage_spells['target_count'] == 1].nlargest(10, 'avg_damage')
print(top_single[['spell_name', 'level', 'damage_dice', 'avg_damage']].to_string(index=False))

\n=== Top 10 Single-Target Damage Spells ===
                    spell_name  level damage_dice  avg_damage
                   Time Ravage      9       10d12        65.0
               Finger of Death      7    7d8 + 30        61.5
                          Harm      6        14d6        49.0
            Psychic Crush (UA)      6        12d6        42.0
                 Reality Break      8        6d12        39.0
                        Blight      4         8d8        36.0
Raulothim's Psychic Lance (UA)      4        10d6        35.0
                 Blade Barrier      6        6d10        33.0
         Negative Energy Flood      5        5d12        32.5
               Banishing Smite      5        5d10        27.5


In [70]:
# Calculate total estimated damage (damage × estimated targets)
damage_spells['total_estimated_damage'] = damage_spells['avg_damage'] * damage_spells['estimated_targets']

print("\n=== Top 10 Spells by Total Estimated Damage ===")
print("(avg_damage × estimated_targets)")
top_total = damage_spells.dropna(subset=['total_estimated_damage']).nlargest(10, 'total_estimated_damage')
print(top_total[['spell_name', 'level', 'damage_dice', 'avg_damage', 'estimated_targets', 'total_estimated_damage']].to_string(index=False))


=== Top 10 Spells by Total Estimated Damage ===
(avg_damage × estimated_targets)
               spell_name  level damage_dice  avg_damage  estimated_targets  total_estimated_damage
                   Symbol      7       10d10        55.0               67.9                 3734.50
               Earthquake      8         5d6        17.5              188.5                 3298.75
                 Sunburst      8        12d6        42.0               67.9                 2851.80
       Maddening Darkness      8         8d8        36.0               67.9                 2444.40
Otiluke's Freezing Sphere      6        10d6        35.0               67.9                 2376.50
             Meteor Swarm      9        20d6        70.0               30.2                 2114.00
          Circle of Death      6         8d6        28.0               67.9                 1901.20
           Call Lightning      3        3d10        16.5               67.9                 1120.35
           Conjure

In [71]:
# Combine damage and target info for damage spells
damage_spells = df[df['avg_damage'] > 0].copy()

print(f"=== Damage Spells: {len(damage_spells)} ===")
print(f"\nAoE damage spells: {damage_spells['is_aoe'].sum()}")
print(f"Single-target damage: {(damage_spells['target_count'] == 1).sum()}")
print(f"Multi-target damage: {((damage_spells['target_count'] > 1) & ~damage_spells['is_aoe']).sum()}")

=== Damage Spells: 210 ===

AoE damage spells: 97
Single-target damage: 94
Multi-target damage: 7


In [72]:
# Final summary
print("\n=== Final Summary ===")
print(f"Total spells: {len(df)}")
print(f"With damage: {(df['avg_damage'] > 0).sum()}")
print(f"AoE: {df['is_aoe'].sum()}")
print(f"Single-target: {(df['target_count'] == 1).sum()}")
print(f"Multi-target (non-AoE): {((df['target_count'] > 1) & ~df['is_aoe']).sum()}")
print(f"\nNew columns added: target_count, is_aoe, aoe_type, aoe_size, estimated_targets")


=== Final Summary ===
Total spells: 574
With damage: 210
AoE: 161
Single-target: 316
Multi-target (non-AoE): 28

New columns added: target_count, is_aoe, aoe_type, aoe_size, estimated_targets


In [73]:
# Top AoE damage spells
print("\n=== Top 10 AoE Damage Spells ===")
top_aoe = damage_spells[damage_spells['is_aoe']].nlargest(10, 'avg_damage')
print(top_aoe[['spell_name', 'level', 'aoe_type', 'aoe_size', 'damage_dice', 'avg_damage']].to_string(index=False))


=== Top 10 AoE Damage Spells ===
                 spell_name  level aoe_type aoe_size damage_dice  avg_damage
               Disintegrate      6     cube  10-foot   10d6 + 40        75.0
               Meteor Swarm      9   radius  40-foot        20d6        70.0
                     Symbol      7   radius  60-foot       10d10        55.0
Abi-Dalzim's Horrid Wilting      8     cube  30-foot        12d8        54.0
           Incendiary Cloud      8   radius  20-foot        10d8        45.0
     Delayed Blast Fireball      7   radius  20-foot        12d6        42.0
                   Sunburst      8   radius  60-foot        12d6        42.0
                 Fire Storm      7     cube  10-foot        7d10        38.5
               Cone of Cold      5     cone  60-foot         8d8        36.0
             Conjure Volley      5   radius  40-foot         8d8        36.0


In [74]:
# Top single-target damage spells
print("\n=== Top 10 Single-Target Damage Spells ===")
top_single = damage_spells[damage_spells['target_count'] == 1].nlargest(10, 'avg_damage')
print(top_single[['spell_name', 'level', 'damage_dice', 'avg_damage']].to_string(index=False))


=== Top 10 Single-Target Damage Spells ===
                    spell_name  level damage_dice  avg_damage
                   Time Ravage      9       10d12        65.0
               Finger of Death      7    7d8 + 30        61.5
                          Harm      6        14d6        49.0
            Psychic Crush (UA)      6        12d6        42.0
                 Reality Break      8        6d12        39.0
                        Blight      4         8d8        36.0
Raulothim's Psychic Lance (UA)      4        10d6        35.0
                 Blade Barrier      6        6d10        33.0
         Negative Energy Flood      5        5d12        32.5
               Banishing Smite      5        5d10        27.5


# Save Updated Data

In [75]:
# Save the updated dataframe
output_path = f"{DATA_DIR}/spells.csv"
df.to_csv(output_path, index=False)
print(f"Saved {len(df)} spells to {output_path}")
print(f"\nColumns: {list(df.columns)}")

Saved 574 spells to ./data/spells.csv

Columns: ['spell_name', 'source', 'level', 'school', 'casting_time', 'range', 'components', 'duration', 'description', 'level_scaling', 'spell_lists', 'damage_dice', 'damage_type', 'avg_damage', 'target_count', 'is_aoe', 'aoe_type', 'aoe_size', 'estimated_targets', 'base_dpr', 'effective_targets', 'modified_dpr', 'inflicts_blinded', 'inflicts_charmed', 'inflicts_deafened', 'inflicts_frightened', 'inflicts_incapacitated', 'inflicts_paralyzed', 'inflicts_petrified', 'inflicts_poisoned', 'inflicts_prone', 'inflicts_restrained', 'inflicts_stunned', 'grants_flying', 'ac_bonus', 'attack_bonus', 'save_bonus', 'grants_advantage', 'inflicts_disadvantage', 'grants_flying.1', 'ac_bonus.1', 'attack_bonus.1', 'save_bonus.1', 'grants_advantage.1', 'inflicts_disadvantage.1', 'grants_flying.2', 'ac_bonus.2', 'attack_bonus.2', 'save_bonus.2', 'grants_advantage.2', 'inflicts_disadvantage.2', 'grants_flying.3', 'ac_bonus.3', 'attack_bonus.3', 'save_bonus.3', 'grants

In [76]:
# Final summary
print("\n=== Final Summary ===")
print(f"Total spells: {len(df)}")
print(f"With damage: {(df['avg_damage'] > 0).sum()}")
print(f"AoE: {df['is_aoe'].sum()}")
print(f"Single-target: {(df['target_count'] == 1).sum()}")
print(f"Multi-target: {(df['target_count'] > 1).sum()}")


=== Final Summary ===
Total spells: 574
With damage: 210
AoE: 161
Single-target: 316
Multi-target: 28


# Merge Spellcasters

Determine which features a creature gains by having spells

In [77]:

spell_feature_columns = ['spell_name', 'level', 
       'modified_dpr', 'inflicts_blinded',
       'inflicts_charmed', 'inflicts_deafened', 'inflicts_frightened',
       'inflicts_incapacitated', 'inflicts_paralyzed', 'inflicts_petrified',
       'inflicts_poisoned', 'inflicts_prone', 'inflicts_restrained',
       'inflicts_stunned', 'grants_flying', 'ac_bonus', 'attack_bonus',
       'save_bonus', 'grants_advantage', 'inflicts_disadvantage',
       ]
df_spells_joinable = df[spell_feature_columns].copy()
df_spells_joinable['spell_name'] = df_spells_joinable['spell_name'].apply(lambda x: x.lower())
# Re-runs can cause duplicate col names
df_spells_joinable = df_spells_joinable.loc[:, ~df_spells_joinable.columns.duplicated()]


In [78]:
load_path = f"{DATA_DIR}/spellcasters_spells.csv"
expanded_spellcasters_raw = pd.read_csv(load_path)



In [79]:
expanded_spellcasters = expanded_spellcasters_raw.merge(
    df_spells_joinable,
    left_on='spell',
    right_on='spell_name',
    how='left'
).drop(columns=['spell_name'])

In [80]:
for col in expanded_spellcasters.columns:
    if expanded_spellcasters[col].dtype == 'bool':
        expanded_spellcasters[col] = expanded_spellcasters[col].fillna(False)
    if expanded_spellcasters[col].dtype == 'int64':
        expanded_spellcasters[col] = expanded_spellcasters[col].fillna(0.0)
    if expanded_spellcasters[col].dtype == 'float64':
        expanded_spellcasters[col] = expanded_spellcasters[col].fillna(0.0)        

df_spellcaster_features = expanded_spellcasters.copy()
df_spellcaster_features = df_spellcaster_features.drop(columns=['spell'])
df_spellcaster_features = df_spellcaster_features.groupby('Name').max().reset_index()

def spelllvl_to_caster_lvl(lvl):
    if lvl == 0:
        return 0
    if lvl == 1:
        return 1
    return 2 * lvl - 1


df_spellcaster_features['spellcaster_level'] = df_spellcaster_features['level'].apply(lambda x: spelllvl_to_caster_lvl(x))
df_spellcaster_features.drop('level', axis=1, inplace=True)
df_spellcaster_features

,Name,modified_dpr,inflicts_blinded,inflicts_charmed,inflicts_deafened,inflicts_frightened,inflicts_incapacitated,inflicts_paralyzed,inflicts_petrified,inflicts_poisoned,inflicts_prone,inflicts_restrained,inflicts_stunned,grants_flying,ac_bonus,attack_bonus,save_bonus,grants_advantage,inflicts_disadvantage,spellcaster_level
0,Acolyte,4.5,False,False,False,False,False,False,False,False,False,False,False,False,0.0,2.5,2.5,False,False,1.0
1,Androsphinx,26.6,False,False,False,False,False,True,False,False,False,True,False,False,0.0,0.0,0.0,False,False,11.0
2,Archmage,144.0,False,False,False,False,False,False,False,False,False,False,False,True,3.0,0.0,0.0,True,False,17.0
3,Cloud Giant,0.0,False,False,False,False,False,False,False,False,False,False,False,True,0.0,0.0,0.0,False,False,15.0
4,Couatl,10.5,False,False,False,False,False,False,False,True,False,False,False,False,5.0,2.5,2.5,False,False,9.0
5,Cult Fanatic,16.5,False,False,False,False,False,True,False,False,False,False,False,False,2.0,0.0,0.0,False,False,3.0
6,Deep Gnome (Svirfneblin),0.0,True,False,True,False,False,False,False,False,False,False,False,False,0.0,0.0,0.0,True,True,5.0
7,Deva,0.0,False,False,False,False,False,False,False,False,False,False,False,False,0.0,0.0,0.0,False,False,9.0
8,Djinni,12.6,False,False,False,False,True,False,False,False,False,False,False,True,0.0,0.0,0.0,False,False,13.0
9,Drider,0.0,False,False,False,False,False,False,False,False,False,False,False,False,0.0,0.0,0.0,True,False,3.0


In [81]:
save_path = DATA_DIR + '/spellcaster_spell_features.csv'
df_spellcaster_features.to_csv(save_path, index=False)

In [82]:
expanded_spellcasters[expanded_spellcasters['Name']=='Ice Mephit']

,Name,spell,level,modified_dpr,inflicts_blinded,inflicts_charmed,inflicts_deafened,inflicts_frightened,inflicts_incapacitated,inflicts_paralyzed,...,inflicts_poisoned,inflicts_prone,inflicts_restrained,inflicts_stunned,grants_flying,ac_bonus,attack_bonus,save_bonus,grants_advantage,inflicts_disadvantage
162,Ice Mephit,fog cloud,1.0,0.0,False,False,False,False,False,False,...,False,False,False,False,False,0.0,0.0,0.0,False,False
